In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"

import unsloth
from unsloth import FastLanguageModel
import torch
import torch.nn as nn
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTConfig, SFTTrainer
import json
import random
from collections import Counter

import torch.utils.checkpoint as ckpt
import torch.nn.functional as F

random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/shivraj-pg/miniconda3/envs/stableenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
HYPERPARAMS = {
    "MODEL_NAME": "unsloth/phi-4",
    "MAX_LEN": 1536, #=3×512,  # Max len for prompt, hi, sa in gitapress final is 1529 
    "LOAD_IN_4BIT": True,
    "BATCH_SIZE": 16,
    "GRAD_ACC": 4,
    "EPOCHS": 5,
    "LR": 2.4e-4,
    "LOG_STEPS": 5,
    "SAVE_STEPS": 15,
    "SAVE_LIMIT": 50,
    "EVAL_STEPS": 15,
    "WEIGHT_DECAY": 0.00465,
    "WARMUP_RATIO": 0.05,
    "MAX_GRAD_NORM": 1.0,

    "LORA_R": 32,
    "LORA_ALPHA": 64,
    "LORA_DROPOUT": 0.1,

    "ES_THRESHOLD": 0.001,
    "ES_PATIENCE": 5,

    "DATA_FILE_PATH": "Files/v3_gitapress_final.csv",
    "OUTPUT_DIR": "Trained_Models/Phi4-14B-customLoss",
}

os.makedirs(HYPERPARAMS["OUTPUT_DIR"], exist_ok=True)


In [3]:
# ---------------------------------------------------------------------------
# Fixed meter -> id mapping (Step 1 of the spec)
# ---------------------------------------------------------------------------
METER_TO_ID = {
    "Anuṣṭubh": 0,
    "Vasantatilakā": 1,
    "Śārdūlavikrīḍita": 2,
    "Indravajrā": 3,
    "Sragdharā": 4,
    "Vaṃśastha": 5,
    "Śikhariṇī": 6,
    "Upendravajrā": 7,
    "Mālinī": 8,
    "Śālinī": 9,
}
ID_TO_METER = {v: k for k, v in METER_TO_ID.items()}
NUM_METERS = len(METER_TO_ID)  # M = 10

print(ID_TO_METER)
print(NUM_METERS)

{0: 'Anuṣṭubh', 1: 'Vasantatilakā', 2: 'Śārdūlavikrīḍita', 3: 'Indravajrā', 4: 'Sragdharā', 5: 'Vaṃśastha', 6: 'Śikhariṇī', 7: 'Upendravajrā', 8: 'Mālinī', 9: 'Śālinī'}
10


In [4]:
# ---------------------------------------------------------------------------
# Model / tokenizer (unchanged)
# ---------------------------------------------------------------------------



model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HYPERPARAMS["MODEL_NAME"],
    max_seq_length=HYPERPARAMS["MAX_LEN"],
    load_in_4bit=HYPERPARAMS["LOAD_IN_4BIT"]
)

model = FastLanguageModel.get_peft_model(
    model,
    r=HYPERPARAMS["LORA_R"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj", ],
    lora_alpha=HYPERPARAMS["LORA_ALPHA"],
    lora_dropout=HYPERPARAMS["LORA_DROPOUT"],
    bias="none",
    use_gradient_checkpointing=True,
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

if tokenizer.pad_token_id is None:
    # Needed because we now pad per-batch ourselves (packing is disabled, see below)
    tokenizer.pad_token = tokenizer.eos_token

tokenizer = get_chat_template(tokenizer, chat_template="phi-4")

==((====))==  Unsloth 2026.7.3: Fast Llama patching. Transformers: 5.14.1.
   \\   /|    NVIDIA L40. Num GPUs = 1. Max memory: 44.392 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 363/363 [00:02<00:00, 148.28it/s]
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.3 patched 40 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
# ---------------------------------------------------------------------------
# Dataset (unchanged split logic)
# ---------------------------------------------------------------------------
ds = load_dataset('csv', data_files=HYPERPARAMS["DATA_FILE_PATH"])["train"]

train_ds = ds.filter(lambda x: x["split"] == "train")
val_ds = ds.filter(lambda x: x["split"] == "val")
print(f"Train: {len(train_ds)}")
print(f"Val: {len(val_ds)}")

# ---------------------------------------------------------------------------
# Sanity check: verify meter_cd strings match METER_TO_ID before anything else
# ---------------------------------------------------------------------------
unique_meters_train = sorted(set(train_ds["meter_cd"]))
print("--------------------------------------------------------------------------------------")
print("Unique meter_cd values found in TRAIN split:")
for m in unique_meters_train:
    print(f"  '{m}'")
print("--------------------------------------------------------------------------------------")

unknown_meters = [m for m in unique_meters_train if m not in METER_TO_ID]
if unknown_meters:
    raise ValueError(
        f"Found meter_cd values in the training split that are not in METER_TO_ID: {unknown_meters}. "
        f"Fix METER_TO_ID (exact string match, including diacritics) before proceeding."
    )

# Also verify val doesn't contain meters we have no id for (would break meter_id lookup)
unknown_val_meters = [m for m in set(val_ds["meter_cd"]) if m not in METER_TO_ID]
if unknown_val_meters:
    raise ValueError(f"Found meter_cd values in the val split not in METER_TO_ID: {unknown_val_meters}")

Train: 23346
Val: 2918
--------------------------------------------------------------------------------------
Unique meter_cd values found in TRAIN split:
  'Anuṣṭubh'
  'Indravajrā'
  'Mālinī'
  'Sragdharā'
  'Upendravajrā'
  'Vasantatilakā'
  'Vaṃśastha'
  'Śikhariṇī'
  'Śālinī'
  'Śārdūlavikrīḍita'
--------------------------------------------------------------------------------------


In [6]:
# ---------------------------------------------------------------------------
# Step 2: meter counts (train split only) and sqrt-inverse-frequency weights
# ---------------------------------------------------------------------------
meter_counts = Counter(train_ds["meter_cd"])  # computed from actual data, not hardcoded

weights_raw = {}
for meter_name, meter_id in METER_TO_ID.items():
    n_m = meter_counts.get(meter_name, 0)
    if n_m <= 0:
        raise ValueError(
            f"Meter '{meter_name}' has zero examples in the training split; "
            f"1/sqrt(N_m) is undefined. Check METER_TO_ID / the dataset."
        )
    weights_raw[meter_name] = 1.0 / (n_m ** 0.5)

mean_raw = sum(weights_raw.values()) / NUM_METERS  # (1/M) * sum_j w_j^raw

meter_weights = {m: (weights_raw[m] / mean_raw) for m in METER_TO_ID}

print("--------------------------------------------------------------------------------------")
print(f"{'Meter':<20}{'Count':<10}{'Weight':<10}")
print("-" * 50)
for meter_name, meter_id in sorted(METER_TO_ID.items(), key=lambda kv: kv[1]):
    print(f"{meter_name:<20}{meter_counts.get(meter_name, 0):<10}{meter_weights[meter_name]:<10.4f}")
print("-" * 50)
print(f"Mean weight (should be 1.0): {sum(meter_weights.values()) / NUM_METERS:.6f}")
print("--------------------------------------------------------------------------------------")

# Ordered tensor, index == meter_id
meter_weight_tensor = torch.zeros(NUM_METERS, dtype=torch.float32)
for meter_name, meter_id in METER_TO_ID.items():
    meter_weight_tensor[meter_id] = meter_weights[meter_name]


# ---------------------------------------------------------------------------
# Formatting + tokenization + response-only label masking + meter_id
#
# NOTE ON A REQUIRED DEVIATION FROM THE ORIGINAL SCRIPT:
# The original script used `packing=True` together with `train_on_responses_only`.
# Packing concatenates multiple different training examples (each possibly a
# different meter, each with its own T_i) into a single training sequence.
# That is fundamentally incompatible with a per-example meter weight and a
# per-example length-normalized CE term (Step 3/4 of the spec need one scalar
# T_i and one scalar w_m per example). So packing is turned OFF here and we
# tokenize/label each example individually, carrying `meter_id` through in the
# same row so it survives into the batch. This is the minimal change required
# to make the requested loss well-defined; nothing else about the model,
# prompt format, tokenizer, split, LoRA config, optimizer or scheduler changes.
# ---------------------------------------------------------------------------


--------------------------------------------------------------------------------------
Meter               Count     Weight    
--------------------------------------------------
Anuṣṭubh            21766     0.0852    
Vasantatilakā       529       0.5466    
Śārdūlavikrīḍita    212       0.8635    
Indravajrā          174       0.9531    
Sragdharā           166       0.9758    
Vaṃśastha           139       1.0664    
Śikhariṇī           131       1.0985    
Upendravajrā        99        1.2636    
Mālinī              75        1.4518    
Śālinī              55        1.6953    
--------------------------------------------------
Mean weight (should be 1.0): 1.000000
--------------------------------------------------------------------------------------


In [7]:
def format_and_tokenize(example):
    system_msg = {"role": "system", "content": example["prompt"]}
    user_msg = {"role": "user", "content": f"Meaning:\n{example['hi']}\n\nGenerate the Sanskrit verse.\n"}
    assistant_msg = {"role": "assistant", "content": example["sa"]}

    full_convo = [system_msg, user_msg, assistant_msg]
    prompt_only_convo = [system_msg, user_msg]

    full_text = tokenizer.apply_chat_template(
        full_convo, tokenize=False, add_generation_prompt=False
    )
    prompt_text = tokenizer.apply_chat_template(
        prompt_only_convo, tokenize=False, add_generation_prompt=True
    )

    input_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    # Sanity check that the prompt is a true prefix of the full sequence.
    prefix_len = len(prompt_ids)
    if input_ids[:prefix_len] != prompt_ids:
        # Fall back defensively: mask nothing extra, but flag it loudly.
        raise ValueError(
            "Tokenized prompt is not a prefix of the tokenized full conversation. "
            "Chat template / tokenizer mismatch — inspect this example before training."
        )

    if len(input_ids) > HYPERPARAMS["MAX_LEN"]:
        input_ids = input_ids[:HYPERPARAMS["MAX_LEN"]]

    labels = list(input_ids)
    mask_len = min(prefix_len, len(labels))
    for i in range(mask_len):
        labels[i] = -100

    attention_mask = [1] * len(input_ids)
    meter_id = METER_TO_ID[example["meter_cd"]]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "meter_id": meter_id,
    }


keep_cols = ["input_ids", "attention_mask", "labels", "meter_id"]

train_ds_tok = train_ds.map(format_and_tokenize, batched=False)
val_ds_tok = val_ds.map(format_and_tokenize, batched=False)

train_ds_tok = train_ds_tok.remove_columns(
    [c for c in train_ds_tok.column_names if c not in keep_cols]
)
val_ds_tok = val_ds_tok.remove_columns(
    [c for c in val_ds_tok.column_names if c not in keep_cols]
)



In [8]:
print("--------------------------------------------------------------------------------------")
print("Sample tokenized/labeled/meter-tagged training examples:")
sample_idx = random.sample(range(len(train_ds)), k=min(3, len(train_ds)))
for idx in sample_idx:
    row = train_ds[idx]
    print(f"meter_cd: {row['meter_cd']}")
    print(f"meter_id: {METER_TO_ID[row['meter_cd']]}")
    print(f"prompt/message (hi): {row['hi']}")
    print(f"target (sa): {row['sa']}")
    print()
print("--------------------------------------------------------------------------------------")


--------------------------------------------------------------------------------------
Sample tokenized/labeled/meter-tagged training examples:
meter_cd: Anuṣṭubh
meter_id: 0
prompt/message (hi): वेदाध्ययनका सार है सत्यभाषण, सत्यभाषणका सार है इन्द्रियसंयम और इन्द्रियसंयमका फल है मोक्ष यही सम्पूर्ण शास्त्रोंका उपदेश है
target (sa): वेदस्योपनिषत् सत्यं सत्यस्योपनिषद् दमः दमस्योपनिषन्मोक्ष एतत् सर्वानुशासनम् ॥

meter_cd: Anuṣṭubh
meter_id: 0
prompt/message (hi): कुमित्रपर भला विश्वास कैसे हो सकता है और कुदेशमें जीना भी सम्भव नहीं है खोटे राजासे सर्वदा भय बना रहता है और कुपुत्रसे तो सब प्रकारसे दुःख ही मिलता है ॥
target (sa): कुसौहदे क्व विश्वास: कुदेशे न तु जीव्यते कुराजनि भयं नित्यं कुपुत्रे सर्वती 5सुखम्‌॥

meter_cd: Anuṣṭubh
meter_id: 0
prompt/message (hi): भीष्मजी कहते हैं राजन् ! ज्ञानी महात्मा पराशर मुनिके मुखसे इस यथार्थ उपदेशको सुनकर धर्मज्ञोंमें श्रेष्ठ राजा जनक बहुत प्रसन्न हुए
target (sa): इत्युक्तो जनको राजन् याथातथ्यं मनीषिणा श्रुत्वा धर्मविदां श्रेष्ठः परां मुदमवाप ह

------

In [9]:
# ---------------------------------------------------------------------------
# Custom collator: dynamic padding while carrying meter_id through
# ---------------------------------------------------------------------------
class MeterWeightedCollator:
    def __init__(self, tokenizer):
        self.pad_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []
        batch_meter_ids = []

        for f in features:
            ids = f["input_ids"]
            mask = f["attention_mask"]
            labels = f["labels"]
            pad_n = max_len - len(ids)

            batch_input_ids.append(ids + [self.pad_id] * pad_n)
            batch_attention_mask.append(mask + [0] * pad_n)
            batch_labels.append(labels + [-100] * pad_n)
            batch_meter_ids.append(f["meter_id"])

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
            "meter_ids": torch.tensor(batch_meter_ids, dtype=torch.long),
        }


In [10]:
base_model = model.base_model.model

In [11]:
# ---------------------------------------------------------------------------
# Custom trainer: meter-weighted, length-normalized CE (Steps 3-5 of the spec)
# ---------------------------------------------------------------------------
class MeterWeightedSFTTrainer(SFTTrainer):
    def __init__(self, *args, meter_weight_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.meter_weight_tensor = meter_weight_tensor

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        meter_ids = inputs.pop("meter_ids")
        labels = inputs["labels"]
        
        outputs = model.base_model.model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                )
        logits = outputs.logits

        # Standard causal-LM shift
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.size())  # [B, T-1]

        valid_mask = (shift_labels != -100).float()  # [B, T-1]

        # L_i^CE = (1/T_i) * sum_t CE_{i,t}
        sequence_loss = (token_loss * valid_mask).sum(dim=1) / valid_mask.sum(dim=1).clamp_min(1)

        # w_{m_i}
        weights = self.meter_weight_tensor.to(sequence_loss.device, sequence_loss.dtype)[meter_ids]

        # L_i = w_{m_i} * L_i^CE ; L_batch = mean_i L_i
        weighted_sequence_loss = sequence_loss * weights
        loss = weighted_sequence_loss.mean()

        return (loss, outputs) if return_outputs else loss

In [12]:
# ---------------------------------------------------------------------------
# Trainer (packing disabled — see note above; everything else unchanged)
# ---------------------------------------------------------------------------
trainer = MeterWeightedSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    data_collator=MeterWeightedCollator(tokenizer),
    meter_weight_tensor=meter_weight_tensor,
    args=SFTConfig(
        output_dir=HYPERPARAMS["OUTPUT_DIR"],

        per_device_train_batch_size=HYPERPARAMS["BATCH_SIZE"],
        gradient_accumulation_steps=HYPERPARAMS["GRAD_ACC"],
        num_train_epochs=HYPERPARAMS["EPOCHS"],

        learning_rate=HYPERPARAMS["LR"],
        lr_scheduler_type="cosine",
        warmup_ratio=HYPERPARAMS["WARMUP_RATIO"],
        weight_decay=HYPERPARAMS["WEIGHT_DECAY"],
        max_grad_norm=HYPERPARAMS["MAX_GRAD_NORM"],

        logging_steps=HYPERPARAMS["LOG_STEPS"],
        logging_strategy="steps",
        logging_dir=HYPERPARAMS["OUTPUT_DIR"] + "/runs",
        report_to="tensorboard",

        save_steps=HYPERPARAMS["SAVE_STEPS"],
        save_total_limit=HYPERPARAMS["SAVE_LIMIT"],

        eval_strategy="steps",
        eval_steps=HYPERPARAMS["EVAL_STEPS"],

        fp16=False,
        bf16=True,
        
        max_steps=30,

        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        optim="adamw_8bit",
        seed=3407,

        # Required so the `meter_id` metadata column survives into the collator.
        remove_unused_columns=False,
        # We already tokenized/labeled the dataset ourselves above.
        packing=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 23,346 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 131,072,000 of 14,790,579,200 (0.89% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
15,0.456391,0.125569
30,0.495321,0.118266


/home/shivraj-pg/miniconda3/envs/stableenv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/shivraj-pg/miniconda3/envs/stableenv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/shivraj-pg/miniconda3/envs/stableenv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`Attention

TrainOutput(global_step=30, training_loss=0.6003296693166097, metrics={'train_runtime': 4608.5961, 'train_samples_per_second': 0.417, 'train_steps_per_second': 0.007, 'total_flos': 1.593699479912448e+17, 'train_loss': 0.6003296693166097, 'epoch': 0.0821917808219178})

In [15]:
trainer.state.log_history

[{'loss': 1.0642971038818358,
  'grad_norm': 0.5560644865036011,
  'learning_rate': 0.00023699134946181884,
  'epoch': 0.0136986301369863,
  'step': 5},
 {'loss': 0.5937553405761719,
  'grad_norm': 0.257703572511673,
  'learning_rate': 0.0002048528137423857,
  'epoch': 0.0273972602739726,
  'step': 10},
 {'loss': 0.4563908100128174,
  'grad_norm': 0.29233378171920776,
  'learning_rate': 0.00014670251207475772,
  'epoch': 0.0410958904109589,
  'step': 15},
 {'eval_loss': 0.125569149851799,
  'eval_runtime': 1076.2448,
  'eval_samples_per_second': 2.711,
  'eval_steps_per_second': 0.678,
  'epoch': 0.0410958904109589,
  'step': 15},
 {'loss': 0.41384596824645997,
  'grad_norm': 0.21131908893585205,
  'learning_rate': 8.036651256537999e-05,
  'epoch': 0.0547945205479452,
  'step': 20},
 {'loss': 0.5783679485321045,
  'grad_norm': 0.16548210382461548,
  'learning_rate': 2.6180222103836425e-05,
  'epoch': 0.0684931506849315,
  'step': 25},
 {'loss': 0.49532084465026854,
  'grad_norm': 0.259

In [16]:
[x for x in trainer.state.log_history if "eval_loss" in x]

[{'eval_loss': 0.125569149851799,
  'eval_runtime': 1076.2448,
  'eval_samples_per_second': 2.711,
  'eval_steps_per_second': 0.678,
  'epoch': 0.0410958904109589,
  'step': 15},
 {'eval_loss': 0.11826647073030472,
  'eval_runtime': 1074.1562,
  'eval_samples_per_second': 2.717,
  'eval_steps_per_second': 0.68,
  'epoch': 0.0821917808219178,
  'step': 30}]

In [14]:
trainer.save_model(HYPERPARAMS["OUTPUT_DIR"] + "/final_model")
tokenizer.save_pretrained(HYPERPARAMS["OUTPUT_DIR"] + "/final_model")

print("BEST MODEL STATS")
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

essential_config = {
    "HYPER-PARAMETERS": HYPERPARAMS,
    "TRAIN_DATASET_LEN": len(train_ds_tok),
    "VAL_DATASET_LEN": len(val_ds_tok),
    "METER_TO_ID": METER_TO_ID,
    "METER_COUNTS_TRAIN": {m: meter_counts.get(m, 0) for m in METER_TO_ID},
    "METER_WEIGHTS": meter_weights,
    "best_model": {
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "best_model_metric": trainer.state.best_metric,
    },
}

with open(HYPERPARAMS["OUTPUT_DIR"] + "/essential_config.json", "w", encoding="utf-8") as f:
    json.dump(essential_config, f, indent=4, default=str, ensure_ascii=False)

Unsloth: Restored added_tokens_decoder metadata in Trained_Models/Phi4-14B-customLoss/final_model/tokenizer_config.json.


BEST MODEL STATS
Trained_Models/Phi4-14B-customLoss/checkpoint-30
0.11826647073030472
